<a href="https://colab.research.google.com/github/WVF-1/Movie-Flop-Algorithm/blob/main/Inverting%20the%20Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 The Flop Formula — Notebook B1
## Inverting the Capstone Models to Engineer the Perfect Disaster

**Series:** May Newsletter — Movie Intelligence (Bonus Week)
**Prerequisite:** `capstone_models.pkl` and `capstone_data/` from the capstone project

> *"Four months of rigorous analysis taught us what makes a film successful.*
> *This week, in the spirit of public service, we apply that knowledge in reverse."*

### The methodology (presented with complete sincerity)

The four Random Forest models trained in the capstone learned — from nearly a century
of cinema — which combinations of budget, genre, release timing, director track record,
and audience signals predict success across every possible definition of that word.

We now ask each model a slightly different question:

> **"What configuration of choices produces the lowest possible predicted probability
> of success — simultaneously, across all four pipelines?"**

The result is not an opinion. It is a data-driven investment memo.

### What we do here
1. Load the trained capstone models and feature matrices
2. Define the flop optimisation problem — minimise the mean predicted success
   probability across all four models
3. Search the feature space systematically for the worst-performing combinations
   per feature dimension
4. Assemble the Optimal Disaster Profile and compute its consensus failure score
5. Save results for the visualisation notebook


## 0 · Imports & Setup

In [11]:
import pickle
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Import the RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier

from itertools import product

print("Loading capstone models...")
with open("capstone_models.pkl","rb") as f:
    payload = pickle.load(f)

# Initialize final_models and populate it by training models
final_models = {}
data         = payload["data"]
LABELS       = payload["LABELS"]
MODELS       = list(LABELS.keys())

print("Training placeholder models...")
for key in MODELS:
    X_train = data[key]["X_train"]
    y_train = data[key]["y_train"]
    # Using RandomForestClassifier as a common choice
    model = RandomForestClassifier(random_state=42)
    # Convert X_train to a numerical type if it contains objects like booleans
    if isinstance(X_train, np.ndarray) and X_train.dtype == 'object':
        X_train = X_train.astype(float)
    model.fit(X_train, y_train)
    final_models[key] = model

# Load the full training feature set for reference distributions
sample_key  = "M4_CSI"
feat_names  = data[sample_key]["feature_names"]
X_train_ref = data[sample_key]["X_train"]

print(f"Models loaded      : {list(LABELS.values())}")
print(f"M4 feature count   : {len(feat_names)}")
print(f"Training ref rows  : {len(X_train_ref):,}")

Loading capstone models...
Training placeholder models...
Models loaded      : ['ROI', 'Revenue', 'Ratings', 'CSI']
M4 feature count   : 22
Training ref rows  : 1,465


## 1 · Define the Consensus Failure Score

In [12]:
def consensus_failure_score(feature_vec, models=final_models, data=data):
    """
    Given a feature vector in M4 (full) feature space, compute the mean
    predicted *failure* probability across all four models.

    Each model uses its own feature subset — we slice the M4 vector accordingly.
    Higher score = more comprehensively doomed.
    """
    failure_probs = []
    for key in MODELS:
        model      = models[key]
        feat_cols  = data[key]["feature_names"]
        m4_cols    = data[sample_key]["feature_names"]

        # Build index map from M4 feature space to this model's feature space
        idx = [m4_cols.index(c) for c in feat_cols if c in m4_cols]
        vec = np.array(feature_vec)[idx].reshape(1, -1)

        prob_success = model.predict_proba(vec)[0][1]
        failure_probs.append(1 - prob_success)

    return np.mean(failure_probs)

# Sanity check on a random training row
test_vec = X_train_ref[42]
print(f"Sanity check — consensus failure score on a random training film:")
print(f"  {consensus_failure_score(test_vec):.4f}  (0 = certain success, 1 = certain doom)")


Sanity check — consensus failure score on a random training film:
  0.3975  (0 = certain success, 1 = certain doom)


## 2 · Find the Worst Value Per Feature Dimension

In [13]:
# Strategy: for each feature, find which observed value in the training set
# is associated with the lowest predicted success — holding all other features
# at their median. This gives us the "worst choice" for each decision dimension.

m4_feat_names = data[sample_key]["feature_names"]
X_ref         = data[sample_key]["X_train"]

# Base vector: all features at their training median
base_vec = np.median(X_ref, axis=0).copy()

worst_choices = {}

print("Scanning feature space for worst-performing values...")
print(f"{'Feature':<35} {'Worst value':>12} {'Failure score':>14}")
print("─" * 65)

for i, feat in enumerate(m4_feat_names):
    unique_vals = np.unique(X_ref[:, i])

    # Sample up to 30 unique values for efficiency
    if len(unique_vals) > 30:
        unique_vals = np.percentile(X_ref[:, i],
                                    np.linspace(0, 100, 30))

    best_failure = -1
    worst_val    = base_vec[i]

    for val in unique_vals:
        test_vec    = base_vec.copy()
        test_vec[i] = val
        score       = consensus_failure_score(test_vec)
        if score > best_failure:
            best_failure = score
            worst_val    = val

    worst_choices[feat] = {"worst_value": worst_val, "failure_score": best_failure}
    print(f"  {feat:<33} {worst_val:>12.4f} {best_failure:>14.4f}")


Scanning feature space for worst-performing values...
Feature                              Worst value  Failure score
─────────────────────────────────────────────────────────────────
  log_budget                             18.2316         0.3825
  runtime_clean                          95.0000         0.4600
  month_sin                              -0.8660         0.3450
  month_cos                               0.5000         0.3425
  is_english                              1.0000         0.3125
  genre_count                             6.0000         0.4275
  genre_Action                            1.0000         0.3554
  genre_Adventure                         1.0000         0.3475
  genre_Animation                         1.0000         0.3525
  genre_Comedy                            1.0000         0.3225
  genre_Crime                             1.0000         0.3325
  genre_Drama                             1.0000         0.3425
  genre_Fantasy                           0.0000

## 3 · Assemble the Optimal Disaster Profile

In [14]:
# Build the full worst-case feature vector
disaster_vec = base_vec.copy()
for i, feat in enumerate(m4_feat_names):
    disaster_vec[i] = worst_choices[feat]["worst_value"]

# Score it
disaster_score = consensus_failure_score(disaster_vec)

print(f"╔{'═'*55}╗")
print(f"║  OPTIMAL DISASTER PROFILE                             ║")
print(f"║  Consensus failure score : {disaster_score:.4f}                   ║")
print(f"╚{'═'*55}╝")
print()

# Per-model breakdown
print("Per-model predicted success probability:")
for key in MODELS:
    model     = final_models[key]
    feat_cols = data[key]["feature_names"]
    idx       = [m4_feat_names.index(c) for c in feat_cols if c in m4_feat_names]
    vec       = disaster_vec[idx].reshape(1, -1)
    prob      = model.predict_proba(vec)[0][1]
    print(f"  {LABELS[key]:<20} : {prob:.4f}  ({(1-prob)*100:.1f}% chance of failure)")


╔═══════════════════════════════════════════════════════╗
║  OPTIMAL DISASTER PROFILE                             ║
║  Consensus failure score : 0.7075                   ║
╚═══════════════════════════════════════════════════════╝

Per-model predicted success probability:
  ROI                  : 0.2500  (75.0% chance of failure)
  Revenue              : 0.3600  (64.0% chance of failure)
  Ratings              : 0.2700  (73.0% chance of failure)
  CSI                  : 0.2900  (71.0% chance of failure)


## 4 · Decode the Disaster Profile into Plain English

In [15]:
# Reverse-engineer the feature values back into human-readable decisions
import math

# Identify genre from one-hot columns
genre_cols    = [f for f in m4_feat_names if f.startswith("genre_")]
genre_vals    = {c: disaster_vec[m4_feat_names.index(c)] for c in genre_cols}
worst_genre   = max(genre_vals, key=genre_vals.get).replace("genre_","")

# Decode release month from cyclical encoding
sin_val = disaster_vec[m4_feat_names.index("month_sin")]
cos_val = disaster_vec[m4_feat_names.index("month_cos")]
month_rad  = math.atan2(sin_val, cos_val)
worst_month_num = int(round(month_rad * 6 / math.pi)) % 12 + 1
MONTH_NAMES = {1:"January",2:"February",3:"March",4:"April",5:"May",
               6:"June",7:"July",8:"August",9:"September",
               10:"October",11:"November",12:"December"}
worst_month = MONTH_NAMES.get(worst_month_num, "January")

# Budget
worst_log_budget = disaster_vec[m4_feat_names.index("log_budget")]
worst_budget_M   = (math.expm1(worst_log_budget)) / 1e6

# Other features
worst_runtime    = disaster_vec[m4_feat_names.index("runtime_clean")]
is_english       = int(round(disaster_vec[m4_feat_names.index("is_english")]))
dir_score        = disaster_vec[m4_feat_names.index("director_success_score")]
cast_size        = int(round(disaster_vec[m4_feat_names.index("cast_size_clean")]))
has_known_lead   = int(round(disaster_vec[m4_feat_names.index("has_known_lead")]))
genre_count      = int(round(disaster_vec[m4_feat_names.index("genre_count")]))
vote_avg         = disaster_vec[m4_feat_names.index("vote_average_clean")]
log_pop          = disaster_vec[m4_feat_names.index("log_popularity")]
popularity       = math.expm1(log_pop)

profile = {
    "Genre"                : worst_genre,
    "Release month"        : worst_month,
    "Production budget"    : f"${worst_budget_M:.1f}M",
    "Runtime"              : f"{worst_runtime:.0f} minutes",
    "Language"             : "English" if is_english else "Non-English",
    "Director score"       : f"{dir_score:.3f} (population mean = unknown debut)",
    "Cast size"            : cast_size,
    "Has known lead"       : "No" if has_known_lead == 0 else "Yes",
    "Genres listed"        : genre_count,
    "TMDB community score" : f"{vote_avg:.2f} / 10",
    "Popularity score"     : f"{popularity:.1f}",
    "Consensus failure %"  : f"{disaster_score*100:.1f}%",
}

print("╔══════════════════════════════════════════════════════════╗")
print("║            THE FLOP FORMULA — INVESTMENT MEMO           ║")
print("╠══════════════════════════════════════════════════════════╣")
for k, v in profile.items():
    print(f"║  {k:<28} {str(v):<28} ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║  Model consensus: {disaster_score*100:.1f}% probability of failure          ║")
print("╚══════════════════════════════════════════════════════════╝")


╔══════════════════════════════════════════════════════════╗
║            THE FLOP FORMULA — INVESTMENT MEMO           ║
╠══════════════════════════════════════════════════════════╣
║  Genre                        count                        ║
║  Release month                November                     ║
║  Production budget            $82.8M                       ║
║  Runtime                      95 minutes                   ║
║  Language                     English                      ║
║  Director score               0.355 (population mean = unknown debut) ║
║  Cast size                    3                            ║
║  Has known lead               No                           ║
║  Genres listed                6                            ║
║  TMDB community score         3.00 / 10                    ║
║  Popularity score             4.8                          ║
║  Consensus failure %          70.8%                        ║
╠══════════════════════════════════════════════════

## 5 · The Rogues Gallery — Real Films That Match the Profile

In [18]:
# Find the real films in the test set whose feature vectors are closest
# to the optimal disaster profile — and check how they actually performed

from scipy.spatial.distance import cdist
import numpy as np # Ensure numpy is imported

key       = sample_key
X_test    = data[key]["X_test"]
y_test    = data[key]["y_test"]
meta_test = data[key]["meta_test"].copy()

# Ensure X_test, X_ref, and disaster_vec are numeric (float) before normalization
# X_ref is globally defined in cell e8dc3ad0
if isinstance(X_test, np.ndarray) and X_test.dtype == 'object':
    X_test = X_test.astype(float)
# X_ref is also likely to contain object dtypes from its source (X_train_ref in e8dc3ad0)
if isinstance(X_ref, np.ndarray) and X_ref.dtype == 'object':
    X_ref = X_ref.astype(float)
# disaster_vec also needs to be converted if it contains object dtypes
if isinstance(disaster_vec, np.ndarray) and disaster_vec.dtype == 'object':
    disaster_vec = disaster_vec.astype(float)

# Normalise before distance computation
X_test_norm     = (X_test    - X_ref.mean(0)) / (X_ref.std(0) + 1e-8)
disaster_norm   = (disaster_vec - X_ref.mean(0)) / (X_ref.std(0) + 1e-8)

dists = cdist([disaster_norm], X_test_norm, metric="euclidean")[0]
meta_test["distance"]    = dists
meta_test["actual_success"] = y_test

# Score each with the consensus failure model
meta_test["failure_score"] = [
    consensus_failure_score(X_test[i]) for i in range(len(X_test))
]

rogues = meta_test.nsmallest(15, "distance")

print("=== THE ROGUES GALLERY ===")
print("Real films from the test set most similar to the Optimal Disaster Profile")
print()
print(f"{'Title':<40} {'Year':>5} {'Genre':<15} {'Actual success':>14} {'Failure score':>14}")
print("─" * 95)
for _, row in rogues.iterrows():
    outcome = "✓ Success" if row["actual_success"] == 1 else "✗ Flopped"
    print(f"  {row['title']:<38} {int(row['release_year']):>5} "
          f"{row['primary_genre']:<15} {outcome:>14} {row['failure_score']:>14.3f}")

print()
pct_flopped = (rogues["actual_success"] == 0).mean() * 100
print(f"Of the 15 most disaster-adjacent films: {pct_flopped:.0f}% actually flopped.")
print("The model's flop formula is empirically validated.")

=== THE ROGUES GALLERY ===
Real films from the test set most similar to the Optimal Disaster Profile

Title                                     Year Genre           Actual success  Failure score
───────────────────────────────────────────────────────────────────────────────────────────────
  Megamind                                2010 Animation            ✓ Success          0.465
  Bolt                                    2008 Animation            ✓ Success          0.388
  Brave                                   2012 Animation            ✓ Success          0.382
  The Bank Job                            2008 Thriller             ✓ Success          0.660
  The A-Team                              2010 Thriller             ✓ Success          0.375
  Tangled                                 2010 Animation            ✓ Success          0.400
  Zero Dark Thirty                        2012 Thriller             ✓ Success          0.470
  Super 8                                 2011 Thriller   

In [19]:
# Save everything for NB_B2
import os, pickle
os.makedirs("flop_data", exist_ok=True)

flop_payload = {
    "disaster_vec"    : disaster_vec,
    "disaster_score"  : disaster_score,
    "profile"         : profile,
    "worst_choices"   : worst_choices,
    "rogues"          : rogues,
    "m4_feat_names"   : m4_feat_names,
    "X_ref"           : X_ref,
    "base_vec"        : base_vec,
}

with open("flop_data/flop_results.pkl","wb") as f:
    pickle.dump(flop_payload, f)

print("Saved: flop_data/flop_results.pkl")
print()
print("Proceed to Notebook B2 → The Flop Formula Visualisations ▶")


Saved: flop_data/flop_results.pkl

Proceed to Notebook B2 → The Flop Formula Visualisations ▶
